In [119]:
import pandas as pd
import numpy as np

BASE_EXISTING_DATA_PATH='../data/combined/'
BASE_TO_BE_ADDED_DATA_PATH='../data/interim/'
BASE_DATA_PROCESSED_PATH="../data/processed/"

ACTIVE_BANKS = ['ADBL','CZBIL','EBL','GBIME','HBL','KBL','MBL','NABIL','NICA','NMB','PCBL','PRVU','SANIMA','SBL','SCB']
FY_LAG_DAYS = 120

POLICY_RATE = {
    2009:6.5, 2010:7.0, 2011:7.0, 2012:7.0, 2013:7.0, 2014:7.0, 2015:7.0,
    2016:5.0, 2017:5.0, 2018:5.0, 2019:4.5, 2020:4.0, 2021:4.5,
    2022:7.0, 2023:5.5, 2024:5.0, 2025:4.5, 2026:4.5
}


In [120]:
nepse_df=pd.read_csv(BASE_TO_BE_ADDED_DATA_PATH+"nepse_index.csv")
car_df=pd.read_excel(BASE_TO_BE_ADDED_DATA_PATH+"CAR.xlsx")
npl_df=pd.read_excel(BASE_TO_BE_ADDED_DATA_PATH+"NPL.xlsx")

bank_df=pd.read_csv(BASE_EXISTING_DATA_PATH+"combinedBankData.csv")


In [121]:
bank_df.columns=bank_df.columns.str.strip()
bank_df['bank']=bank_df['bank'].str.strip()
bank_df['date']=pd.to_datetime(bank_df['date'].str.strip())
bank_df['per_change']=pd.to_numeric(bank_df['per_change'].str.strip(),errors='coerce')
bank_df=bank_df.drop_duplicates(subset=['bank','date'],keep='last')

bank_df=bank_df.sort_values(['bank','date']).reset_index(drop=True)
bank_df['open']=bank_df['open'].clip(lower=bank_df['low'],upper=bank_df['high'])
bank_df.loc[bank_df['amount']==0,'amount']=np.nan
bank_df.tail()

,date,open,high,low,close,per_change,volume,amount,bank
47183,2026-02-24,640.0,648.0,638.0,648.0,1.25,21552.0,13851424.4,SCB
47184,2026-02-25,649.9,649.9,638.9,642.2,-0.90,14223.0,9160194.8,SCB
47185,2026-02-26,643.1,652.0,643.0,647.1,0.76,24303.0,15761927.3,SCB
47186,2026-03-01,647.0,651.0,641.1,647.2,0.02,14211.0,9201778.3,SCB
47187,2026-03-03,647.2,652.0,642.1,650.0,0.43,9794.0,6356859.7,SCB


In [122]:
nepse_df=(nepse_df.rename(columns={"timestamp":'date','close':'nepse_close'})[['date','nepse_close']]
          .assign(date=lambda x:pd.to_datetime(x['date']))
.sort_values('date'))

nepse_df['nepse_ema200']=nepse_df['nepse_close'].ewm(span=200,min_periods=100).mean()
nepse_df['nepse_bull']=(nepse_df['nepse_close']>nepse_df['nepse_ema200']).astype(int)
nepse_df['nepse_ret_1d']=nepse_df['nepse_close'].pct_change(1)
nepse_df['nepse_ret_5d']=nepse_df['nepse_close'].pct_change(5)
nepse_df['nepse_ret_21d']=nepse_df['nepse_close'].pct_change(21)

bank_df=bank_df.merge(nepse_df.drop(columns='nepse_ema200'), on='date',how='left')
bank_df['policy_rate']=bank_df['date'].dt.year.map(POLICY_RATE)
bank_df.head()

,date,open,high,low,close,per_change,volume,amount,bank,nepse_close,nepse_bull,nepse_ret_1d,nepse_ret_5d,nepse_ret_21d,policy_rate
0,2010-09-16,117.0,122.0,116.0,120.0,NaN,5280.0,NaN,ADBL,404.43,0.0,-0.006949,-0.024130,-0.109656,7.0
1,2010-09-19,120.0,120.0,116.0,118.0,-1.67,2648.0,NaN,ADBL,402.44,0.0,-0.004921,-0.026535,-0.112982,7.0
2,2010-09-20,118.0,119.0,116.0,118.0,0.00,2346.0,NaN,ADBL,401.74,0.0,-0.001739,-0.021697,-0.101594,7.0
3,2010-09-21,118.0,118.0,115.0,116.0,-1.69,7160.0,NaN,ADBL,403.94,0.0,0.005476,-0.015477,-0.087636,7.0
4,2010-09-23,117.0,120.0,117.0,120.0,3.45,8417.0,NaN,ADBL,404.41,0.0,0.001164,-0.006998,-0.085149,7.0


In [123]:
def to_annual_long(df,value_col):
    df=df.copy()
    df.columns=[c.strip() if isinstance(c,str) else c for c in df.columns]
    df['SYMBOL']=df['SYMBOL'].str.strip()
    df=df[df['SYMBOL'].isin(ACTIVE_BANKS)]
    long=df.melt(id_vars='SYMBOL',var_name='fy_year',value_name=value_col)
    long['fy_year']=pd.to_numeric(long['fy_year'],errors="coerce")
    long[value_col]=pd.to_numeric(long[value_col],errors='coerce')
    long=long.dropna(subset=['fy_year',value_col])
    long['date']=pd.to_datetime(long['fy_year'].astype(int).astype(str)+'-07-15')+pd.Timedelta(days=FY_LAG_DAYS)
    return long.rename(columns={'SYMBOL':'bank'})[['bank','date',value_col]].sort_values(['bank','date'])

def merge_annual(base,fund,value_col):
    chunks=[]
    for bank in ACTIVE_BANKS:
        pc=base[base['bank']==bank].sort_values('date')
        fc=fund[fund['bank']==bank].sort_values('date')
        chunks.append(pd.merge_asof(pc,fc[['date',value_col]],on='date',direction='backward'))
    return pd.concat(chunks).sort_values(['bank','date']).reset_index(drop=True)

car_long=to_annual_long(car_df,'car')
npl_long=to_annual_long(npl_df,'npl')

bank_df=merge_annual(bank_df,car_long,'car')
bank_df=merge_annual(bank_df,npl_long,'npl')
bank_df=bank_df.dropna()
bank_df.isna().sum()
bank_df.info()



<class 'pandas.DataFrame'>
Index: 44095 entries, 252 to 47105
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           44095 non-null  datetime64[us]
 1   open           44095 non-null  float64       
 2   high           44095 non-null  float64       
 3   low            44095 non-null  float64       
 4   close          44095 non-null  float64       
 5   per_change     44095 non-null  float64       
 6   volume         44095 non-null  float64       
 7   amount         44095 non-null  float64       
 8   bank           44095 non-null  str           
 9   nepse_close    44095 non-null  float64       
 10  nepse_bull     44095 non-null  float64       
 11  nepse_ret_1d   44095 non-null  float64       
 12  nepse_ret_5d   44095 non-null  float64       
 13  nepse_ret_21d  44095 non-null  float64       
 14  policy_rate    44095 non-null  float64       
 15  car            44095 non-null  fl

In [124]:
bank_df.to_csv(BASE_DATA_PROCESSED_PATH+'combinedBankData.csv',index=False)